In [2]:
#this is likely the endpoint we will use:
from nba_api.stats.endpoints import leaguegamefinder
from sqlalchemy import create_engine
from requests.exceptions import ReadTimeout
import pandas as pd
import random
import time 
import json

#create our engine for pushing to sql database
from dotenv import load_dotenv
import os

#helper functions
def normalize_for_postgres(df):
    """Normalize DataFrame column names for PostgreSQL"""
    df = df.copy()
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    return df

load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

#define seasons to gather data for 
# Season formats - NBA API uses different formats in different endpoints!
seasons = ['2023-24', '2022-23', '2021-22', '2020-21', '2019-20',
           '2018-19', '2017-18', '2016-17', '2015-16', '2014-15',
           '2013-14', '2012-13', '2011-12', '2010-11', '2009-10']

# For PlayerGameLog.SEASON_ID filtering (API internal format: '2YYYY')
seasons_api = ['22023', '22022', '22021', '22020', '22019',
               '22018', '22017', '22016', '22015', '22014',
               '22013', '22012', '22011', '22010', '22009']

all_seasons_data = []

for season in seasons:
    try:
        # Get all games for a season
        game_finder = leaguegamefinder.LeagueGameFinder(
            season_nullable=season,  # Format: YYYY-YY
            league_id_nullable='00',    # 00 is NBA
            season_type_nullable='Regular Season'  # or 'Playoffs'
        )
        
        # Convert to DataFrame
        games_df = game_finder.get_data_frames()[0]
        games_df['SEASON'] = season
        games_df_sorted = games_df.sort_values(['TEAM_ID', 'GAME_DATE'], ascending=[True, False])
        all_seasons_data.append(games_df)

        print(f"✓ Successfully collected {len(games_df)} game records for {season}")
        time.sleep(round(random.uniform(1, 3), 1))

    #handle timeout/api rejection errors
    except (ReadTimeout, json.decoder.JSONDecodeError) as e:
        print(f"Error for {season}: {e} - retrying after 60 seconds")
        time.sleep(60)
        try:
            game_finder = leaguegamefinder.LeagueGameFinder(
                season_nullable=season,
                league_id_nullable='00',
                season_type_nullable='Regular Season'
            )
            games_df = game_finder.get_data_frames()[0]
            games_df['SEASON'] = season
            games_df = games_df.sort_values(['TEAM_ID', 'GAME_DATE'], ascending=[True, False])
            all_seasons_data.append(games_df)
            print(f"✓ Retry successful for {season}")
            time.sleep(round(random.uniform(3, 6), 1))
        except Exception as retry_error:
            print(f"✗ Retry failed for {season}: {retry_error}")
            print(f"  Skipping {season} and continuing...")

# Combine all seasons into one giant dataframe
print("\nCombining all seasons into single dataframe...")
combined_df = pd.concat(all_seasons_data, ignore_index=True)

print(f"Total records collected: {len(combined_df)}")
print(f"Seasons covered: {combined_df['SEASON'].unique()}")
print(f"Teams covered: {combined_df['TEAM_ID'].nunique()}")

# Normalize column names before pushing to database
print("\nNormalizing column names for PostgreSQL...")
combined_df = normalize_for_postgres(combined_df)

# Push to SQL as a single table
print("\nPushing to SQL database...")
combined_df.to_sql('team_game_stats', engine, if_exists='replace', index=False)

print("\n✓ Finished! All data stored in 'team_game_stats' table")

✓ Successfully collected 2460 game records for 2023-24
✓ Successfully collected 2460 game records for 2022-23
✓ Successfully collected 2460 game records for 2021-22
✓ Successfully collected 2160 game records for 2020-21
✓ Successfully collected 2118 game records for 2019-20
✓ Successfully collected 2460 game records for 2018-19
✓ Successfully collected 2460 game records for 2017-18
✓ Successfully collected 2460 game records for 2016-17
✓ Successfully collected 2460 game records for 2015-16
✓ Successfully collected 2460 game records for 2014-15
✓ Successfully collected 2460 game records for 2013-14
✓ Successfully collected 2458 game records for 2012-13
✓ Successfully collected 1980 game records for 2011-12
✓ Successfully collected 2460 game records for 2010-11
✓ Successfully collected 2460 game records for 2009-10

Combining all seasons into single dataframe...
Total records collected: 35776
Seasons covered: ['2023-24' '2022-23' '2021-22' '2020-21' '2019-20' '2018-19' '2017-18'
 '2016-1